In [1]:
import pandas as pd
df=pd.read_csv('amh_final_5_raters_5915.tsv', sep='\t')


In [10]:
import ast

df['scale'] = df['scale'].apply(lambda x: ast.literal_eval(x))

def process_scale(scale_list):
    emotions = {
        'Anger': [],
        'Disgust': [],
        'Fear': [],
        'Sadness': [],
        'Joy': [],
        'Surprise': []
    }
    
    for value in scale_list:
        # Extract the intensity level (last character of the string)
        level = int(value[-1])  
        
        # Check which emotion the value corresponds to and add the intensity level
        if value.startswith('a'):
            emotions['Anger'].append(level)
        elif value.startswith('d'):
            emotions['Disgust'].append(level)
        elif value.startswith('f'):
            emotions['Fear'].append(level)
        elif value.startswith('sa'):
            emotions['Sadness'].append(level)
        elif value.startswith('j'):
            emotions['Joy'].append(level)
        elif value.startswith('su'):
            emotions['Surprise'].append(level)
    
    return pd.Series(emotions)

# Apply the function to create new emotion columns for each row
df = df.join(df['scale'].apply(process_scale))

In [13]:
#emo column is 1=Anger, 2=Disgust, 3=Fear, 4=Sadness, 5=Joy, 6=Surprise, and 7= Neutral(no emotion)
#scale column is the scale of the corosponding emotion
#a1 = low anger
#a2 = medium anger
#a3 = high anger, the same for others sa=sadness, su=surprise, j=joy, d=disgust, and f=fear
df = df[['displayed_text','emo','scale']]
df 

,instance_id,displayed_text,emo,scale,Anger,Disgust,Fear,Sadness,Joy,Surprise,list_length
0,--yLHZ0xprY_an_22056,ዉጋ አስፋልግም እኛ ጦርነት አንፋልግም የኔ ወንድሜ,"[1.0, 1.0, 7.0, 7.0, 7.0, 4.0]","[a2, a1, sa2]","[2, 1]",[],[],[2],[],[],6
1,--yLHZ0xprY_fe_16890,የገንዘብ ጥቅም ፍለጋ አገር ማስበጥበጥ ምን ይጠቅማል የደሃ ልጆችን ማስጨ...,"[1.0, 1.0, 1.0, 7.0, 7.0, 2.0]","[a2, a3, a1, d2]","[2, 3, 1]",[2],[],[],[],[],6
2,--yLHZ0xprY_fe_16893,ጁቡቲ መደንገጥ የለባትም እንደ ጎበዝ ነጋዴ ዜዴ መፍጠር ነው ያለባት 1 ...,"[1.0, 7.0, 7.0, 7.0, 2.0]","[a3, d1]",[3],[1],[],[],[],[],5
3,--yLHZ0xprY_sr_16914,የሚገርም ነው ሀገሬ ጠላትሽ ምነው ቁጥሩ በዛ አይዞሽ ሁሉን ቻዩ እግዚአብ...,"[6.0, 6.0, 4.0, 4.0, 4.0, 4.0]","[su2, su2, sa2, sa2, sa2, sa2]",[],[],[],"[2, 2, 2, 2]",[],"[2, 2]",6
4,--yLHZ0xprY_sr_22052,ወላሂ እኔም ገረመኝ እዝህ ከፋኖ በዛ ከኤርትራ ደሞ በዝህ ከሱማሌ እዳበደ...,"[1.0, 6.0, 6.0, 6.0, 2.0, 2.0]","[a1, d2, d2, su2, su2, su2]",[1],"[2, 2]",[],[],[],"[2, 2, 2]",6
...,...,...,...,...,...,...,...,...,...,...,...
5995,zq5fMxwklKA_sr_10852,ህዝቡ በረሃብ ያልቃል ከተማ ላይ ጭፈራው ደርቷል መንግስት የተራበ ህዝብ ...,"[1.0, 1.0, 1.0, 7.0, 2.0]","[a2, a3, a3, d2]","[2, 3, 3]",[2],[],[],[],[],5
5996,zqEUPF_ob6U_jo_18050,በጣም ጥሩ ነው ማጋራታቸው ነገር ግን እንደ ኢቢኤስ ለምን በጎ ስራ በምግ...,"[5.0, 5.0, 5.0, 5.0, 7.0]","[j2, j2, j3, j1]",[],[],[],[],"[2, 2, 3, 1]",[],5
5997,zsuXxIAFrFo_di_1740,ኧረ አሁንስ ከእንተ ጋር ተደምሮ ኢትዮጵያዊ መባል ዘጋኝ አሳቀቀኝም ኤጭ ...,"[1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0]","[a2, a3, a3, a1, d2, d2, d3]","[2, 3, 3, 1]","[2, 2, 3]",[],[],[],[],7
5998,zsuXxIAFrFo_sd_1742,ኤትዮጵያ ታንቀሽ ሙቺ ያለ የለም ከፈለገች ግን ታንቃ ኣደለም በቁምዋ ብት...,"[1.0, 1.0, 2.0, 2.0, 2.0]","[a2, a2, d2, d2, d3]","[2, 2]","[2, 2, 3]",[],[],[],[],5


In [ ]:
df.reset_index(drop=True, inplace=True)
df.rename(columns={'instance_id': 'id', 'displayed_text':'text'}, inplace=True)
df = df[['id','text','scale','Anger','Disgust','Fear','Sadness','Joy','Surprise']]
df

## Track 1, multi-label emotion

In [ ]:
# final emotion label
def convert_to_binary(emotion_list):
    if not emotion_list:  # Empty list
        return 0
    avg = sum(emotion_list) / 5#len(emotion_list)
    annotators  = sum(1 for i in emotion_list)
    
    if avg >= 0.6 and annotators  >= 2:
        return 1
    else:
        return 0

# Iterate over each emotion column and apply the conversion function
for emotion in ['Anger', 'Disgust', 'Fear', 'Sadness', 'Joy', 'Surprise']:
    df[emotion] = df[emotion].apply(convert_to_binary)
# df

In [16]:
df = df[['text',	'scale', 'Anger','Disgust','Fear','Sadness','Joy','Surprise']]
df

,text,scale,Anger,Disgust,Fear,Sadness,Joy,Surprise
0,ዉጋ አስፋልግም እኛ ጦርነት አንፋልግም የኔ ወንድሜ,"[a2, a1, sa2]",1,0,0,0,0,0
1,የገንዘብ ጥቅም ፍለጋ አገር ማስበጥበጥ ምን ይጠቅማል የደሃ ልጆችን ማስጨ...,"[a2, a3, a1, d2]",1,0,0,0,0,0
2,ጁቡቲ መደንገጥ የለባትም እንደ ጎበዝ ነጋዴ ዜዴ መፍጠር ነው ያለባት 1 ...,"[a3, d1]",0,0,0,0,0,0
3,የሚገርም ነው ሀገሬ ጠላትሽ ምነው ቁጥሩ በዛ አይዞሽ ሁሉን ቻዩ እግዚአብ...,"[su2, su2, sa2, sa2, sa2, sa2]",0,0,0,1,0,1
4,ወላሂ እኔም ገረመኝ እዝህ ከፋኖ በዛ ከኤርትራ ደሞ በዝህ ከሱማሌ እዳበደ...,"[a1, d2, d2, su2, su2, su2]",0,1,0,0,0,1
...,...,...,...,...,...,...,...,...
5910,ህዝቡ በረሃብ ያልቃል ከተማ ላይ ጭፈራው ደርቷል መንግስት የተራበ ህዝብ ...,"[a2, a3, a3, d2]",1,0,0,0,0,0
5911,በጣም ጥሩ ነው ማጋራታቸው ነገር ግን እንደ ኢቢኤስ ለምን በጎ ስራ በምግ...,"[j2, j2, j3, j1]",0,0,0,0,1,0
5912,ኧረ አሁንስ ከእንተ ጋር ተደምሮ ኢትዮጵያዊ መባል ዘጋኝ አሳቀቀኝም ኤጭ ...,"[a2, a3, a3, a1, d2, d2, d3]",1,1,0,0,0,0
5913,ኤትዮጵያ ታንቀሽ ሙቺ ያለ የለም ከፈለገች ግን ታንቃ ኣደለም በቁምዋ ብት...,"[a2, a2, d2, d2, d3]",1,1,0,0,0,0


In [17]:
emotion_columns = ['Anger', 'Disgust', 'Fear','Sadness','Joy','Surprise']
from sklearn.model_selection import StratifiedShuffleSplit

# Combine emotion columns into one for stratification
y = df[emotion_columns].idxmax(axis=1)

# StratifiedShuffleSplit
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.4)

for train_idx, temp_idx in splitter.split(df, y):
    train_df = df.iloc[train_idx]
    temp_df = df.iloc[temp_idx]

# Further split temp into test and dev
splitter_temp = StratifiedShuffleSplit(n_splits=1, test_size=0.25)

for test_idx, dev_idx in splitter_temp.split(temp_df, temp_df[emotion_columns].idxmax(axis=1)):
    test_df = temp_df.iloc[test_idx]
    dev_df = temp_df.iloc[dev_idx]

In [ ]:
train_df.reset_index(drop=True, inplace=True)
prefix_train = "amh_train_track1_"
train_df['id'] = prefix_train + (train_df.index + 1).astype(str)
train_df = train_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
train_df.to_csv('amh_track1/amh_train_track1.tsv',sep='\t', index=False)


test_df.reset_index(drop=True, inplace=True)
prefix_test= "amh_test_track1_"
test_df['id'] = prefix_test + (test_df.index + 1).astype(str)
test_df = test_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
test_df.to_csv('amh_track1/amh_test_track1.tsv',sep='\t', index=False)

dev_df.reset_index(drop=True, inplace=True)
prefix_dev = "amh_dev_track1_"
dev_df['id'] = prefix_dev + (dev_df.index + 1).astype(str)
dev_df = dev_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
dev_df.to_csv('amh_track1/amh_dev_track1.tsv',sep='\t', index=False)


## Track 2, multi-label intensity

In [ ]:
# intensity 0, 1, 2, 3
def convert_to_category(emotion_list):
    if not emotion_list:  # Empty list
        return 0
    avg = sum(emotion_list) / 5#len(emotion_list)
    annotators  = sum(1 for i in emotion_list)
    
    if avg >= 0.6 and avg < 1.5 and annotators  >= 2:
        return 1
    elif avg >= 1.5 and avg < 2.5 and annotators  >= 2:
        return 2
    elif avg >= 2.5 and annotators  >= 2:
        return 3
    else:
        return 0

# Iterate over each emotion column and apply the conversion function
for emotion in ['Anger', 'Disgust', 'Fear', 'Sadness', 'Joy', 'Surprise']:
    df[emotion] = df[emotion].apply(convert_to_category)

In [23]:
df = df[['id','text','scale','Anger','Disgust','Fear','Sadness','Joy','Surprise']]
df


,id,text,scale,Anger,Disgust,Fear,Sadness,Joy,Surprise
0,--yLHZ0xprY_an_22056,ዉጋ አስፋልግም እኛ ጦርነት አንፋልግም የኔ ወንድሜ,"[a2, a1, sa2]",1,0,0,0,0,0
1,--yLHZ0xprY_fe_16890,የገንዘብ ጥቅም ፍለጋ አገር ማስበጥበጥ ምን ይጠቅማል የደሃ ልጆችን ማስጨ...,"[a2, a3, a1, d2]",1,0,0,0,0,0
2,--yLHZ0xprY_fe_16893,ጁቡቲ መደንገጥ የለባትም እንደ ጎበዝ ነጋዴ ዜዴ መፍጠር ነው ያለባት 1 ...,"[a3, d1]",0,0,0,0,0,0
3,--yLHZ0xprY_sr_16914,የሚገርም ነው ሀገሬ ጠላትሽ ምነው ቁጥሩ በዛ አይዞሽ ሁሉን ቻዩ እግዚአብ...,"[su2, su2, sa2, sa2, sa2, sa2]",0,0,0,2,0,1
4,--yLHZ0xprY_sr_22052,ወላሂ እኔም ገረመኝ እዝህ ከፋኖ በዛ ከኤርትራ ደሞ በዝህ ከሱማሌ እዳበደ...,"[a1, d2, d2, su2, su2, su2]",0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...
5910,zq5fMxwklKA_sr_10852,ህዝቡ በረሃብ ያልቃል ከተማ ላይ ጭፈራው ደርቷል መንግስት የተራበ ህዝብ ...,"[a2, a3, a3, d2]",2,0,0,0,0,0
5911,zqEUPF_ob6U_jo_18050,በጣም ጥሩ ነው ማጋራታቸው ነገር ግን እንደ ኢቢኤስ ለምን በጎ ስራ በምግ...,"[j2, j2, j3, j1]",0,0,0,0,2,0
5912,zsuXxIAFrFo_di_1740,ኧረ አሁንስ ከእንተ ጋር ተደምሮ ኢትዮጵያዊ መባል ዘጋኝ አሳቀቀኝም ኤጭ ...,"[a2, a3, a3, a1, d2, d2, d3]",2,1,0,0,0,0
5913,zsuXxIAFrFo_sd_1742,ኤትዮጵያ ታንቀሽ ሙቺ ያለ የለም ከፈለገች ግን ታንቃ ኣደለም በቁምዋ ብት...,"[a2, a2, d2, d2, d3]",1,1,0,0,0,0


In [24]:
emotion_columns = ['Anger', 'Disgust', 'Fear','Sadness','Joy','Surprise']
from sklearn.model_selection import StratifiedShuffleSplit

# Combine emotion columns into one for stratification
y = df[emotion_columns].idxmax(axis=1)

# StratifiedShuffleSplit
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.4)

for train_idx, temp_idx in splitter.split(df, y):
    train_df = df.iloc[train_idx]
    temp_df = df.iloc[temp_idx]

# Further split temp into test and dev
splitter_temp = StratifiedShuffleSplit(n_splits=1, test_size=0.25)

for test_idx, dev_idx in splitter_temp.split(temp_df, temp_df[emotion_columns].idxmax(axis=1)):
    test_df = temp_df.iloc[test_idx]
    dev_df = temp_df.iloc[dev_idx]

In [ ]:
train_df.reset_index(drop=True, inplace=True)
prefix_train = "amh_train_track2_"
train_df['id'] = prefix_train + (train_df.index + 1).astype(str)
train_df = train_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
train_df.to_csv('amh_track2/amh_train_track2.tsv',sep='\t', index=False)


test_df.reset_index(drop=True, inplace=True)
prefix_test= "amh_test_track2_"
test_df['id'] = prefix_test + (test_df.index + 1).astype(str)
test_df = test_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
test_df.to_csv('amh_track2/amh_test_track2.tsv',sep='\t', index=False)

dev_df.reset_index(drop=True, inplace=True)
prefix_dev = "amh_dev_track2_"
dev_df['id'] = prefix_dev + (dev_df.index + 1).astype(str)
dev_df = dev_df[['id', 'text', 'Anger', 'Disgust', 'Fear', 'Sadness', 'Joy','Surprise']]
dev_df.to_csv('amh_track2/amh_dev_track2.tsv',sep='\t', index=False)
